# SMS Spam Dataset - Hugging Face'e Yükleme

Bu notebook, Sebastian Raschka'nın kitabındaki SMS Spam veri setini Hugging Face'e yükler.

## 1. Kurulum

In [1]:
# Gerekli kütüphaneleri yükle
!pip install pandas numpy requests datasets huggingface_hub -q

## 2. Hugging Face Girişi

In [2]:
from google.colab import userdata
from huggingface_hub import login

# Colab Secrets'tan HF_TOKEN al
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("✅ Hugging Face'e giriş yapıldı!")

✅ Hugging Face'e giriş yapıldı!


## 3. Veri Setini İndir ve İşle

In [3]:
import pandas as pd
import requests
import zipfile
import os
from pathlib import Path

# Veri setini indir
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"

if not data_file_path.exists():
    print("Veri seti indiriliyor...")
    response = requests.get(url, stream=True, timeout=60)
    response.raise_for_status()
    with open(zip_path, "wb") as out_file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                out_file.write(chunk)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path = Path(extracted_path) / "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"Veri seti kaydedildi: {data_file_path}")
else:
    print("Veri seti zaten mevcut.")

Veri seti indiriliyor...
Veri seti kaydedildi: sms_spam_collection/SMSSpamCollection.tsv


In [4]:
# DataFrame'e yükle
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
print(f"Toplam örnek: {len(df)}")
print(f"\nLabel dağılımı:")
print(df["Label"].value_counts())

Toplam örnek: 5572

Label dağılımı:
Label
ham     4825
spam     747
Name: count, dtype: int64


In [5]:
# Balanced dataset oluştur (her class'tan eşit sayıda)
def create_balanced_dataset(df):
    num_spam = df[df["Label"] == "spam"].shape[0]
    ham_subset = df[df["Label"] == "ham"].sample(num_spam, random_state=123)
    balanced_df = pd.concat([ham_subset, df[df["Label"] == "spam"]])
    return balanced_df

balanced_df = create_balanced_dataset(df)
print(f"Balanced dataset boyutu: {len(balanced_df)}")
print(balanced_df["Label"].value_counts())

Balanced dataset boyutu: 1494
Label
ham     747
spam    747
Name: count, dtype: int64


In [6]:
# Label'ları sayısal değere çevir (ham=0, spam=1)
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})
print(balanced_df.head())

      Label                                               Text
4307      0  Awww dat is sweet! We can think of something t...
4138      0                             Just got to  &lt;#&gt;
4831      0  The word "Checkmate" in chess comes from the P...
4461      0  This is wishing you a great day. Moji told me ...
5440      0      Thank you. do you generally date the brothas?


In [7]:
# Train/Validation/Test split (70/10/20)
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

print(f"Eğitim: {len(train_df)}")
print(f"Doğrulama: {len(validation_df)}")
print(f"Test: {len(test_df)}")

Eğitim: 1045
Doğrulama: 149
Test: 300


## 4. Hugging Face Dataset Oluştur

In [8]:
from datasets import Dataset

# Pandas DataFrame'leri Hugging Face Dataset'e çevir
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
validation_dataset = Dataset.from_pandas(validation_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# DatasetDict oluştur
from datasets import DatasetDict

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset
})

print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['Label', 'Text'],
        num_rows: 1045
    })
    validation: Dataset({
        features: ['Label', 'Text'],
        num_rows: 149
    })
    test: Dataset({
        features: ['Label', 'Text'],
        num_rows: 300
    })
})


In [10]:
print(dataset_dict)

DatasetDict({
    train: Dataset({
        features: ['Label', 'Text'],
        num_rows: 1045
    })
    validation: Dataset({
        features: ['Label', 'Text'],
        num_rows: 149
    })
    test: Dataset({
        features: ['Label', 'Text'],
        num_rows: 300
    })
})


## 5. Hugging Face'e Yükle

In [13]:
# Dataset'i Hugging Face'e yükle
dataset_name = "Mustafaege/sms-spam-balanced"

dataset_dict.push_to_hub(
    dataset_name,
    token=hf_token
)

print(f"✅ Dataset yüklendi: https://huggingface.co/datasets/{dataset_name}")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 77.3kB / 77.3kB            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 24.2kB / 24.2kB            

✅ Dataset yüklendi: https://huggingface.co/datasets/Mustafaege/sms-spam-balanced
